# ClaimScope on Colab

This notebook runs ClaimScope end to end: it downloads a paper from arXiv, extracts its
claims, decides which are testable, designs a reduced experiment, **actually runs it**, and
issues a verdict with a confidence interval.

## Before you start: the sandbox

On your own machine, ClaimScope runs generated code inside a Docker container with no network, a
non-root user and a read-only root. **Colab has no Docker daemon**, so a weaker backend is used
here: a subprocess with network calls blocked, memory and CPU capped, and the environment
scrubbed.

That **blocks accidental downloads but does not contain hostile code**. It is acceptable here
because the code is generated by a model from a scientific paper, inside a disposable Colab VM,
and because the point is to watch the pipeline work. Do not use it this way for code you do not
trust.

The report produced at the end says so explicitly.

## 1. Installation

In [ ]:
# Unos 2 minutos.
!git clone --quiet https://github.com/USUARIO/claimscope.git 2>/dev/null || echo "ya clonado"
%cd claimscope
!pip install --quiet -e . 2>&1 | tail -2
print("listo")

If you do not have the repository on GitHub, upload the project folder to Colab
(folder icon → upload) and skip the previous cell, running only `pip install -e .` against it.

## 2. The model key

ClaimScope works with Anthropic or with Google Gemini. Gemini has a free tier: get a key at
[aistudio.google.com/apikey](https://aistudio.google.com/apikey).

The key is requested with `getpass`, so it is never written into the notebook.

In [ ]:
import getpass
import os

provider = "google"  # "google" o "anthropic"

if provider == "google":
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")
    os.environ["CLAIMSCOPE_MODEL_NAME"] = "gemini-3.6-flash"
else:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

os.environ["CLAIMSCOPE_PROVIDER"] = provider

# El sandbox de subproceso, porque Colab no tiene Docker.
os.environ["CLAIMSCOPE_SANDBOX_BACKEND"] = "subprocess"

# Un presupuesto pequeño: Colab da CPUs modestas.
os.environ["CLAIMSCOPE_BUDGET_MINUTES_TOTAL"] = "15"
os.environ["CLAIMSCOPE_MAX_CLAIMS"] = "1"

print(f"proveedor: {provider}")

## 3. Check the sandbox works

Before spending any model calls, verify that the sandbox runs code and that it **really blocks
the network**. If this fails, nothing else is meaningful.

In [ ]:
from pathlib import Path

from claimscope.config import get_settings
from claimscope.sandbox.runner import ExecutionRequest, build_runner

settings = get_settings()
runner = build_runner(settings)
print(f"backend: {type(runner).__name__}")

ws = Path("/tmp/sandbox_check")
ws.mkdir(exist_ok=True)

# 1. ¿Ejecuta código?
(ws / "run.py").write_text("print('el sandbox ejecuta codigo')")
r = runner.execute(ExecutionRequest(workspace=ws, args=[], timeout_s=30))
print(f"  run: {'OK' if r.ok else 'FAILED'}  {r.stdout.strip()}")

# 2. ¿Bloquea la red? Esto es lo que importa.
(ws / "run.py").write_text(
    "import socket, sys\n"
    "try:\n"
    "    socket.create_connection(('1.1.1.1', 53), timeout=5)\n"
    "    print('NETWORK REACHABLE -- the block is not working')\n"
    "except OSError as e:\n"
    "    print(f'red bloqueada: {str(e)[:60]}')\n"
    "    sys.exit(3)\n"
)
r = runner.execute(ExecutionRequest(workspace=ws, args=[], timeout_s=30))
blocked = r.exit_code == 3
print(f"  network block: {'OK' if blocked else 'FAILED'}  {r.stdout.strip()}")

assert blocked, "the network block is not working; do not continue"
print("\nsandbox verificado")

## 4. Analyse a paper

Pick a paper with a small, testable claim. The dropout one is ideal: its central claim is that
dropout reduces overfitting in a small network on MNIST, and that fits in CPU minutes.

Other options annotated in the repository:

| arXiv | Paper | Why |
|---|---|---|
| `1207.0580` | Dropout | Clean ablation, cheap |
| `1708.04552` | Cutout | A single knob, on CIFAR |
| `1502.03167` | Batch Normalization | Convergence speed |
| `1907.11692` | RoBERTa | **Nothing testable** — checks that triage can say no |

In [ ]:
PAPER = "1207.0580"

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)

from langgraph.checkpoint.memory import InMemorySaver

from claimscope.graph import build_graph
from claimscope.session import thread_config

graph = build_graph(settings, checkpointer=InMemorySaver())
config = thread_config(f"colab-{PAPER}")

state = graph.invoke({"paper_id": PAPER}, config)

print(f"\n{state.get('paper_title', '')}\n")
for claim in state.get("claims", []):
    mark = "SI " if claim.testable else "no "
    print(f"[{mark}] {claim.claim_type:14} {claim.id}")
    print(f"       {claim.text[:90]}")
    if claim.triage_reason:
        print(f"       -> {claim.triage_reason[:90]}")

print(f"\nseleccionadas: {state.get('selected_claim_ids')}")

## 5. Review the plan

The agent stops here on purpose. Read the reduction plan: what it changes from the paper, what it
preserves, and why the effect should survive the shrinking.

**This is the part most worth your attention.** A plan that changes two things at once, or that
does not keep both arms identical except in what the claim is about, produces a result that means
nothing.

In [ ]:
import json

interrupts = state.get("__interrupt__")
if not interrupts:
    print("Nothing to review: triage selected no claims.")
else:
    for request in interrupts[0].value["requests"]:
        plan = request["plan"]
        print(f"=== {request['claim_id']} ===")
        print(f"{request['claim_text']}\n")
        print(f"Original: {plan['original_setup']}")
        print(f"Reducido: {plan['reduced_setup']}\n")
        print("Cambios:")
        for change in plan["changes"]:
            print(f"  - {change}")
        print("\nPreservado:")
        for item in plan["preserved"]:
            print(f"  - {item}")
        print(f"\nPor que deberia transferirse:\n  {plan['why_claim_should_transfer']}")
        print(f"\nSemillas: {plan['seeds']}  ~{plan['estimated_minutes']:.0f} min")

## 6. Approve and run

If the plan convinces you, approve it. From here the agent generates the experiment's code,
measures it with a short run, checks it fits the budget, and runs it.

It takes a few minutes. If the generated code fails, the `debug` node fixes it and retries, up to
three times.

In [ ]:
from langgraph.types import Command

if interrupts:
    decisions = [
        {"claim_id": r["claim_id"], "action": "approve"} for r in interrupts[0].value["requests"]
    ]
    state = graph.invoke(Command(resume=decisions), config)

print("run finished\n")
for claim_id, results in (state.get("run_results") or {}).items():
    print(f"{claim_id}: {len(results)} runs")
    for r in sorted(results, key=lambda x: (x.arm, x.seed)):
        print(f"  {r.arm:20} semilla {r.seed}  metrica {r.metric_value:.4f}  {r.runtime_s:.0f}s")

for claim_id, reason in (state.get("abandoned_claim_ids") or {}).items():
    print(f"\nabandonada {claim_id}: {reason}")

## 7. The code the model wrote

Worth reading. It is what actually ran.

In [ ]:
for claim_id, workspace in (state.get("workspace_dirs") or {}).items():
    script = Path(workspace) / "run.py"
    if script.exists():
        print(f"===== {claim_id} / run.py =====\n")
        print(script.read_text(encoding="utf-8"))

## 8. The verdict and the report

The effect is the difference of means between arms, signed in the direction the paper predicts.
The confidence interval comes from a bootstrap over the seeds.

- Interval entirely in the predicted direction → `consistent_at_reduced_scale`
- Interval entirely in the opposite one → `not_consistent_at_reduced_scale`
- Interval crossing zero → `inconclusive`

With three seeds the power is low: `inconclusive` almost always means "not enough evidence", not
"no effect".

In [ ]:
for verdict in state.get("verdicts", []):
    print(f"=== {verdict.claim_id} ===")
    print(f"verdict: {verdict.verdict}")
    if verdict.effect_estimate is not None:
        print(
            f"efecto:    {verdict.effect_estimate:+.4g}  "
            f"IC95 [{verdict.ci_low:+.4g}, {verdict.ci_high:+.4g}]  p={verdict.p_value:.3g}"
        )
    print(f"{verdict.notes}\n")

report_path = state.get("report_path")
if report_path:
    print(f"\nfull report at {report_path}\n")
    print("=" * 70)
    print(Path(report_path).read_text(encoding="utf-8"))

## 9. What you have just checked

If you have got this far with a verdict, you have verified that:

- Ingestion and claim extraction work on a real paper.
- Triage tells the testable apart from what is not.
- The reduction plan respects the invariants (same arms, same seeds, same budget).
- The generated code **actually runs** and produces a metric.
- The statistical analysis and the report close the loop.

What you have **not** checked: that the sandbox contains hostile code, because this backend
cannot. That needs Docker.

### Trying the rest

```python
# The subprocess sandbox's containment tests (no Docker needed).
!python -m pytest tests/test_subprocess_runner.py -v

# The full suite.
!python -m pytest -q

# The evaluation harness over the 13 annotated papers.
!python -m eval.run_eval --paper 1207.0580
```